In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 수집, 탐색

In [2]:
df = pd.read_csv(r'C:\Users\Admin\hipython\data\premium.csv', encoding='utf-8')
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500



# 전처리

In [3]:
# 중복체크
df[df.duplicated(keep=False)]

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [4]:
df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1333 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


# 분할
bmi의 Null 처리

In [5]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

# 문자열 데이터의 수치화 > LabelEncoder

In [6]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
col_list = ['sex', 'smoker', 'region']
for col in col_list: 
  enc = LabelEncoder()
  df[col] = enc.fit_transform(df[col])
  
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


# 스케일링 하기
age, sex, bmi, children, smoker, region 스케일링
charges는 타겟값 로짓변환을 수행한다.

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, random_state=42)

NameError: name 'X' is not defined

In [ ]:
from sklearn.preprocessing import StandardScaler

In [9]:
# 연속형 컬럼
scaler = StandardScaler()
X_train['bmi'] = scaler.fit_transform(X_train[['bmi']])
X_test['bmi'] = scaler.transform(X_test[['bmi']])
#X_test['bmi']

NameError: name 'StandardScaler' is not defined

# 스탠다드 스케일링

In [10]:
# ── Standard Scaling : bmi, charges 컬럼만 적용 ──────────────────

from sklearn.preprocessing import StandardScaler

# 1. StandardScaler 객체 생성
std_scaler = StandardScaler()

# 2. bmi, charges 컬럼만 선택
cols = ['bmi', 'charges']

# 3. 훈련 데이터로 fit → transform
X_train_std = std_scaler.fit_transform(df.loc[X_train.index, cols])

# 4. 테스트 데이터는 transform만
X_test_std  = std_scaler.transform(df.loc[X_test.index, cols])

# 5. 결과 확인
print("=== StandardScaler 적용 결과 (bmi, charges) ===")
result_df = pd.DataFrame(X_train_std, columns=[f'{c}_scaled' for c in cols])
print(result_df.head(5))
print()
print(f"평균 (≈0에 가까워야 함): \n{result_df.mean().round(4)}")
print()
print(f"표준편차 (≈1에 가까워야 함): \n{result_df.std().round(4)}")

NameError: name 'X_train' is not defined

# 선형회귀

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [12]:
# 선형회귀 모델
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
mae = mean_squared_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae, mse, rmse, r2

NameError: name 'X_train' is not defined

# 다항회귀
 Degree

In [13]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [14]:
degree = [2,3,4]

for deg in degree:
  model_poly = Pipeline( [('poly', PolynomialFeatures(degree=deg, include_bias=False)),
           ('linear', LinearRegression())])
  model_poly.fit(X_train, y_train)
  poly_pred = model_poly.predict(X_test)

  mae = mean_absolute_error(y_test, poly_pred)
  mse = mean_squared_error(y_test, poly_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, poly_pred)

  print(f'Degree {deg} MAE : {mae:.4f} MSE : {mse:.4f} R^2 : {r2:.4f}')


NameError: name 'X_train' is not defined

In [15]:
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state=0)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)
mae, mse, rmse, r2


NameError: name 'X_train' is not defined

In [16]:
# 특성 중요도
import pandas as pd
feature_names = ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
importance_df = pd.DataFrame({
  'feaure': feature_names,
  'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)

print('\n--- 특성 중요도 ----')
print(importance_df)

NotFittedError: This RandomForestRegressor instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

# XGBRegressor

In [17]:
# XGBRegressor
from xgboost import XGBRegressor

# 모델 선언 및 학습
xgb_model = XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.1)
xgb_model.fit(X_train, y_train)

# 예측
y_pred_xgb = xgb_model.predict(X_test)

# 평가
mae  = mean_absolute_error(y_test, y_pred_xgb)
mse  = mean_squared_error(y_test, y_pred_xgb)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred_xgb)

print(f'MAE  : {mae:.4f}')
print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R²   : {r2:.4f}')

NameError: name 'X_train' is not defined

In [18]:
# 특성 중요도
feature_names = ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
xgb_importance_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\n--- XGBRegressor 특성 중요도 ---')
print(xgb_importance_df)

NotFittedError: need to call fit or load_model beforehand

In [19]:
# 전체 모델 성능 비교 요약
summary = pd.DataFrame({
    '모델'  : ['LinearRegression', 'Poly degree=2', 'Poly degree=3', 'Poly degree=4',
               'RandomForest', 'XGBRegressor'],
    'R²'   : [r2_score(y_test, lr_model.predict(X_test)),
               None, None, None,          # Degree는 루프 안에서만 출력됨
               r2_score(y_test, y_pred_rf),
               r2]
})
print('\n=== 모델별 R² 비교 ===')
print(summary.dropna())

NameError: name 'y_test' is not defined

# 하이퍼파라미터

In [20]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'random_state': 0
    }
    model = RandomForestRegressor(**params)
    score = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f'Best MAE : {round(study.best_value, 4)}')
print(f'Best Params : {study.best_params}')

[I 2026-03-03 11:38:11,776] A new study created in memory with name: no-name-dfd14bce-e791-4516-8b48-215917e5bb4b
[W 2026-03-03 11:38:11,776] Trial 0 failed with parameters: {'n_estimators': 288, 'max_depth': 12, 'min_samples_split': 9, 'min_samples_leaf': 1} because of the following error: NameError("name 'X' is not defined").
Traceback (most recent call last):
  File "c:\Users\Admin\miniconda3\envs\ml_edu\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_20932\3107089446.py", line 14, in objective
    score = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()
                                    ^
NameError: name 'X' is not defined
[W 2026-03-03 11:38:11,776] Trial 0 failed with value None.


NameError: name 'X' is not defined

In [ ]:
best_model = RandomForestRegressor(**study.best_params)
best_model.fit(X_train, y_train)

y_pred_best = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best)
mse = mean_squared_error(y_test, y_pred_best)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_xgb)

print(f'RandomForest (Optuna) - MAE: {round(mae, 4)} / MSE: {round(mse, 4)}')
print(f'RandomForest (Optuna) - rmse: {round(rmse, 4)} / r2: {round(r2, 4)}')

RandomForest (Optuna) - MAE: 2429.099 / MSE: 18613751.9814
RandomForest (Optuna) - rmse: 4314.3658 / r2: 0.8691
